# 01 - Lakehouse Ingestion (Bronze Layer)

**Fabric analogue:** Dataflow Gen2 / Pipeline landing raw files into a Lakehouse's `Tables/` area as Delta tables.

This notebook reads CSVs from `data/raw/` and writes them unchanged (plus ingestion metadata)
to Delta Lake at `data/bronze/`. Bronze = raw + traceable.

In [1]:
import polars as pl
from deltalake import write_deltalake, DeltaTable
from pathlib import Path
from datetime import datetime, timezone
import shutil

RAW = Path('../data/raw')
BRONZE = Path('../data/bronze')
BRONZE.mkdir(parents=True, exist_ok=True)
print('Raw files:', [p.name for p in sorted(RAW.glob('*.csv'))])

Raw files: ['customers_2025_01.csv', 'customers_2025_04.csv', 'products.csv', 'sales.csv']


## Write each raw CSV to a bronze Delta table
We add `_ingested_at` and `_source_file` columns — the same lineage metadata
a Fabric pipeline run would inject.

In [2]:
def ingest(csv_name: str, table_name: str):
    src = RAW / csv_name
    target = BRONZE / table_name
    df = pl.read_csv(src, infer_schema_length=1000)
    df = df.with_columns([
        pl.lit(datetime.now(timezone.utc).isoformat()).alias('_ingested_at'),
        pl.lit(csv_name).alias('_source_file'),
    ])
    # overwrite for idempotent reruns
    if target.exists():
        shutil.rmtree(target)
    write_deltalake(str(target), df.to_arrow(), mode='overwrite')
    return df.height

counts = {
    'bronze_sales':      ingest('sales.csv', 'bronze_sales'),
    'bronze_customers_jan': ingest('customers_2025_01.csv', 'bronze_customers_jan'),
    'bronze_customers_apr': ingest('customers_2025_04.csv', 'bronze_customers_apr'),
    'bronze_products':   ingest('products.csv', 'bronze_products'),
}
counts

{'bronze_sales': 410,
 'bronze_customers_jan': 80,
 'bronze_customers_apr': 80,
 'bronze_products': 30}

## Verify: read one bronze table back as a Delta table

In [3]:
dt = DeltaTable(str(BRONZE / 'bronze_sales'))
pl.from_arrow(dt.to_pyarrow_table()).head(5)

sale_id,customer_id,store_id,product_sku,sale_ts,quantity,unit_price,payment_method,_ingested_at,_source_file
i64,i64,str,str,str,i64,f64,str,str,str
80,76,"""S01""","""SKU-013""","""2025-03-14 17:29:00""",1,63.42,"""cash""","""2026-04-24T17:32:07.151800+00:…","""sales.csv"""
156,10,"""S03""","""SKU-017""","""2025-02-05 08:18:00""",2,92.57,"""wallet""","""2026-04-24T17:32:07.151800+00:…","""sales.csv"""
102,36,"""S01""","""SKU-002""","""2025-01-09 18:53:00""",3,128.81,"""cash""","""2026-04-24T17:32:07.151800+00:…","""sales.csv"""
221,28,"""S01""","""SKU-019""","""2025-04-14 01:19:00""",3,38.21,"""card""","""2026-04-24T17:32:07.151800+00:…","""sales.csv"""
83,66,"""S04""","""SKU-024""","""2025-01-08 23:18:00""",2,182.42,"""card""","""2026-04-24T17:32:07.151800+00:…","""sales.csv"""


Bronze ingestion complete. Note: the `sales` table intentionally contains
duplicates and nulls (`payment_method`) — we will clean these in Silver.